# Manual walkthrough: `setup_chl` + chlorophyll `MOM_override` params

Exercises the new code on branch `move-chl-from-mom6-forge` (PRs [CROCODILE-CESM#69](https://github.com/CROCODILE-CESM/regional-mom6/pull/69) / [COSIMA#313](https://github.com/COSIMA/regional-mom6/pull/313)), which isn't merged yet, so this notebook points `sys.path` at the worktree directly instead of relying on the `mom6-regional-dev` env's editable install (which still points at the original regional-mom6 checkout).

Run this with the `mom6-regional-dev` kernel.

In [ ]:
import sys

WORKTREE = "/glade/u/home/manishrv/documents/croc/regional_mom_workflows/regional-mom6-chl-worktree"
sys.path.insert(0, WORKTREE)

import shutil
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr

import regional_mom6

print(regional_mom6.__file__)
assert regional_mom6.__file__.startswith(WORKTREE), "regional_mom6 did not resolve to the worktree - check sys.path"

from regional_mom6 import experiment
from regional_mom6 import MOM_parameter_tools as mpt

## 1. Build a small experiment

Same synthetic setup as `tests/test_expt_class.py::test_setup_bathymetry` — a small rectangular grid with `hgrid_type="even_spacing"` (auto-generates `m6f_hgrid`/`m6f_vgrid` on construction). `boundaries=[]` and `tidal_constituents=[]` so we skip OBC/tides plumbing and focus on chlorophyll.

In [ ]:
tmp_path = Path(tempfile.mkdtemp(prefix="chl_test_"))
mom_run_dir = tmp_path / "rundir"
mom_input_dir = tmp_path / "inputdir"
print("Working dir:", tmp_path)

longitude_extent = (-5, 5)
latitude_extent = (0, 10)

expt = experiment(
    longitude_extent=longitude_extent,
    latitude_extent=latitude_extent,
    date_range=["2003-01-01 00:00:00", "2003-01-02 00:00:00"],
    resolution=0.5,
    number_vertical_layers=5,
    layer_thickness_ratio=1,
    depth=1000,
    mom_run_dir=mom_run_dir,
    mom_input_dir=mom_input_dir,
    fre_tools_dir="toolpath",
    hgrid_type="even_spacing",
    expt_name="chl_test_grid",
    boundaries=[],
    tidal_constituents=[],
)
expt

## 2. Set up bathymetry

Random synthetic bathymetry (same pattern as the existing bathymetry test) — `setup_chl` needs `topo.tmask` to mask/fill the interpolated chlorophyll.

In [ ]:
bathymetry_file = tmp_path / "bathymetry.nc"

bathymetry = np.random.random((100, 100)) * (-100)
bathymetry = xr.DataArray(
    bathymetry,
    dims=["silly_lat", "silly_lon"],
    coords={
        "silly_lat": np.linspace(latitude_extent[0] - 5, latitude_extent[1] + 5, 100),
        "silly_lon": np.linspace(longitude_extent[0] - 5, longitude_extent[1] + 5, 100),
    },
)
bathymetry.name = "silly_depth"
bathymetry.to_netcdf(bathymetry_file)
bathymetry.close()

expt.setup_bathymetry(
    bathymetry_path=str(bathymetry_file),
    longitude_coordinate_name="silly_lon",
    latitude_coordinate_name="silly_lat",
    vertical_coordinate_name="silly_depth",
)

## 3. Run `setup_chl`

Uses the real SeaWiFS climatology already staged on GLADE. This is the same file `tests/test_chl.py` in mom6_forge points at.

In [ ]:
SEAWIFS_PATH = "/glade/campaign/cesm/cesmdata/cseg/inputdata/ocn/mom/croc/chl/data/SeaWIFS.L3m.MC.CHL.chlor_a.0.25deg.nc"

chl_ds = expt.setup_chl(processed_seawifs_path=SEAWIFS_PATH)
chl_ds

In [ ]:
expected_output = mom_input_dir / f"seawifs-clim-1997-2010-{expt.expt_name}.nc"
assert expected_output.exists(), f"Expected chl output at {expected_output}"
print("Wrote:", expected_output)

chl_ds["CHL_A"].isel(TIME=0).plot()

## 4. Confirm `setup_generic` wires the chlorophyll `MOM_override` params

`setup_generic` normally runs after `setup_fms_version`/`setup_rOM3` seed `mom_run_dir` from the packaged `demos/premade_run_directories/common_files`. We replicate that seeding here directly, then call `setup_generic` and check the resulting `MOM_override` for `CHL_FILE`, `CHL_FROM_FILE`, `VAR_PEN_SW`, `PEN_SW_NBANDS`.

In [ ]:
common_files = Path(WORKTREE) / "demos" / "premade_run_directories" / "common_files"
shutil.copytree(common_files, mom_run_dir, dirs_exist_ok=True)

expt.setup_generic(ncpus=1, mask_land_cpus=False)

In [ ]:
MOM_override_dict = mpt.read_MOM_file_as_dict("MOM_override", mom_run_dir)
for key in ["CHL_FILE", "CHL_FROM_FILE", "VAR_PEN_SW", "PEN_SW_NBANDS"]:
    print(key, "=", MOM_override_dict[key]["value"])

assert MOM_override_dict["CHL_FILE"]["value"] == f'"{expected_output.name}"'
assert MOM_override_dict["CHL_FROM_FILE"]["value"] == "True"
assert MOM_override_dict["VAR_PEN_SW"]["value"] == "True"
assert str(MOM_override_dict["PEN_SW_NBANDS"]["value"]) == "3"
print("\nAll chlorophyll MOM_override params set correctly.")

## 5. Cleanup

In [ ]:
shutil.rmtree(tmp_path)